# Hamiltonians: TFI and Heisenberg 1D

This notebook walks through the key physics of the two new Hamiltonians:  (transverse-field Ising) and . Check each cell by hand — unit tests verify correctness mechanically, but these checks build intuition.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from qaravan.core import TFI, Heisenberg1D, LinearLattice, PauliString, Magnetization
from qaravan.backends.statevector import Statevector

## 1. LinearLattice

In [ ]:
lat_open = LinearLattice(6, periodic=False)
lat_ring = LinearLattice(6, periodic=True)
print("Open:", lat_open.nn_pairs())
print("Ring:", lat_ring.nn_pairs())

## 2. TFI: ground state

**Check:** At =0$, =1$ open chain, ground state is ferromagnetically ordered. Energy  -(n-1)$.

In [ ]:
for n in [3, 4, 5, 6]:
    tfi = TFI(n, J=1.0, h=0.0)
    sv = tfi.ground_state()
    E = sv.expectation(tfi.as_observable()).real
    print(f"n={n}: E={E:.6f}  expected={-(n-1)}")

**Check:** As $ increases through the QPT at /J = 1$ (open chain, thermodynamic limit), ground energy decreases and magnetization drops.

In [ ]:
n = 8
h_values = np.linspace(0.0, 3.0, 30)
energies = []
mags = []
for h in h_values:
    tfi = TFI(n, J=1.0, h=h)
    sv = tfi.ground_state()
    energies.append(sv.expectation(tfi.as_observable()).real)
    mags.append(sv.expectation(Magnetization(n, axis="Z")).real)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(h_values, energies)
axes[0].set_xlabel("h/J"); axes[0].set_ylabel("Ground energy / J")
axes[0].set_title("TFI ground energy")
axes[1].plot(h_values, mags)
axes[1].axvline(1.0, ls="--", color="gray", label="QPT (h/J=1)")
axes[1].set_xlabel("h/J"); axes[1].set_ylabel(r"$\langle M_z angle$")
axes[1].set_title("TFI Z-magnetization")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. TFI: Trotter circuit

**Check:** One-step Trotter error scales as (dt^2)$ (order 1) and (dt^3)$ (order 2). Log-log plot should show straight lines with those slopes.

In [ ]:
n, J, h = 5, 1.0, 0.5
tfi = TFI(n, J=J, h=h)
dt_values = np.array([0.1, 0.07, 0.05, 0.03, 0.02, 0.01])
H_mat = tfi.as_matrix()

err1, err2 = [], []
for dt in dt_values:
    exact = expm(-1j * dt * H_mat)
    err1.append(np.linalg.norm(tfi.trotter_circuit(dt, order=1).to_matrix() - exact, "fro"))
    err2.append(np.linalg.norm(tfi.trotter_circuit(dt, order=2).to_matrix() - exact, "fro"))

plt.figure(figsize=(6, 4))
plt.loglog(dt_values, err1, "o-", label=f"order-1 (slope={np.polyfit(np.log(dt_values), np.log(err1), 1)[0]:.2f})")
plt.loglog(dt_values, err2, "s-", label=f"order-2 (slope={np.polyfit(np.log(dt_values), np.log(err2), 1)[0]:.2f})")
plt.xlabel("dt"); plt.ylabel("Frobenius error")
plt.title(f"TFI Trotter error scaling (n={n})")
plt.legend(); plt.tight_layout(); plt.show()

## 4. Heisenberg 1D: ground state

**Check:** =0$, >0$:  = -h\sum_i Z_i$, ground state $|00\ldots0angle$, energy  -hn$.

In [ ]:
n = 5
heis_j0 = Heisenberg1D(n, J=0.0, h=1.0)
sv = heis_j0.ground_state()
E = sv.expectation(heis_j0.as_observable()).real
print(f"E = {E:.6f},  expected = {-float(n):.6f}")
print("State (should be |00000>):", np.round(sv.to_array(), 4))

**Check:** =1$, =0$, periodic BC. For even $, the XXX model has a ground state in the \mathrm{tot}=0$ (singlet) sector. The energy matches Bethe-ansatz numerics. Compare our -based result to a literature reference:

| $ |  / J$ (Bethe ansatz, PBC) |
|---|---|
| 4 | 569Xl4 \ln 2 + 1 pprox -1.773$ (per bond; total =  	imes (-\ln 2 + 1/4) pprox -1.773 	imes 4/4$) |

In [ ]:
for n in [4, 6, 8]:
    heis = Heisenberg1D(n, J=1.0, h=0.0, periodic=True)
    evals = np.linalg.eigvalsh(heis.as_matrix())
    sv = heis.ground_state()
    E = sv.expectation(heis.as_observable()).real
    # Bethe ansatz per-site energy for PBC XXX chain: e0/J → -ln(2) + 1/4 as n→∞
    # Finite-size values approach this from above.
    print(f"n={n}: E0={E:.6f}, E0/n={E/n:.6f}  (BA limit: {-np.log(2)+0.25:.6f})")

## 5. Heisenberg 1D: Trotter error scaling

**Check:** Same log-log slope test as TFI.

In [ ]:
n, J, h = 5, 1.0, 0.3
heis = Heisenberg1D(n, J=J, h=h)
H_mat = heis.as_matrix()

err1, err2 = [], []
for dt in dt_values:
    exact = expm(-1j * dt * H_mat)
    err1.append(np.linalg.norm(heis.trotter_circuit(dt, order=1).to_matrix() - exact, "fro"))
    err2.append(np.linalg.norm(heis.trotter_circuit(dt, order=2).to_matrix() - exact, "fro"))

plt.figure(figsize=(6, 4))
plt.loglog(dt_values, err1, "o-", label=f"order-1 (slope={np.polyfit(np.log(dt_values), np.log(err1), 1)[0]:.2f})")
plt.loglog(dt_values, err2, "s-", label=f"order-2 (slope={np.polyfit(np.log(dt_values), np.log(err2), 1)[0]:.2f})")
plt.xlabel("dt"); plt.ylabel("Frobenius error")
plt.title(f"Heisenberg Trotter error scaling (n={n})")
plt.legend(); plt.tight_layout(); plt.show()

## 6. Trotter time evolution

**Check:** Evolve a product state under TFI for one full period, measuring $\langle M_z angle$ at each step.

In [ ]:
n, J, h, dt, n_steps = 6, 1.0, 1.5, 0.05, 80
tfi = TFI(n, J=J, h=h)
step_circ = tfi.trotter_circuit(dt, order=2)

sv = Statevector(bitstring="0" * n)
mag_obs = Magnetization(n, "Z")
times, mags = [0.0], [sv.expectation(mag_obs).real]
for step in range(n_steps):
    sv = sv.apply(step_circ)
    times.append((step + 1) * dt)
    mags.append(sv.expectation(mag_obs).real)

plt.figure(figsize=(7, 3.5))
plt.plot(times, mags)
plt.xlabel("t (J=1 units)")
plt.ylabel(r"$\langle M_z angle$")
plt.title(f"TFI Trotter dynamics (n={n}, h/J={h/J})", fontsize=12)
plt.tight_layout()
plt.show()